# Liquidity Management Monte Carlo Demo in Colab

This notebook is a Colab-ready teaching variant of the local Liquidity Management demo.

Learning goals:

- generate synthetic inflow and outflow transactions
- aggregate them into daily net flows
- run a Monte Carlo liquidity forecast
- interpret mean outcome, VaR, and shortfall probability together


In [ ]:
%pip install faker pandas numpy matplotlib seaborn -q

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker

fake = Faker('en_IN')
Faker.seed(42)
random.seed(42)
np.random.seed(42)
sns.set_theme(style='whitegrid')

def build_transactions(num_rows=4000, mean=500000, stddev=200000):
    rows = []
    for i in range(num_rows):
        tx_type = random.choices(['inflow', 'outflow'], weights=[45, 55])[0]
        adjusted_mean = mean * (1.2 if tx_type == 'inflow' else 0.8)
        amount = max(abs(random.gauss(adjusted_mean, stddev)), 1000)
        rows.append({
            'transaction_date': fake.date_between(start_date='-2y', end_date='today'),
            'transaction_type': tx_type,
            'amount_inr': round(amount, 2)
        })
    df = pd.DataFrame(rows)
    df['transaction_date'] = pd.to_datetime(df['transaction_date'])
    df['net_amount'] = np.where(df['transaction_type'].eq('inflow'), df['amount_inr'], -df['amount_inr'])
    return df

df = build_transactions()
df.head()

In [ ]:
daily_flows = df.groupby('transaction_date', as_index=True)['net_amount'].sum().sort_index()
daily_flows.describe()

In [ ]:
def monte_carlo_liquidity(flows, simulations=500, forecast_days=30, initial_cash=10_000_000):
    mean_flow = flows.mean()
    std_flow = flows.std()
    results = np.zeros((simulations, forecast_days + 1))
    results[:, 0] = initial_cash
    for sim in range(simulations):
        cash = initial_cash
        for day in range(1, forecast_days + 1):
            cash += np.random.normal(mean_flow, std_flow)
            results[sim, day] = cash
    return results

results = monte_carlo_liquidity(daily_flows, simulations=800, forecast_days=30)
final_positions = results[:, -1]

summary = {
    'Mean Final Position': final_positions.mean(),
    '5% VaR': np.percentile(final_positions, 5),
    '1% VaR': np.percentile(final_positions, 1),
    'Probability of Shortfall (%)': (final_positions < 0).mean() * 100
}
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = results[:60]
for path in sample:
    axes[0].plot(path, alpha=0.2, color='#0ea5e9')
axes[0].plot(np.median(results, axis=0), color='#1d4ed8', linewidth=2, label='Median')
axes[0].set_title('Monte Carlo Liquidity Paths')
axes[0].set_xlabel('Forecast Day')
axes[0].set_ylabel('Cash Position (INR)')
axes[0].legend()

axes[1].hist(final_positions, bins=35, color='#22c55e', edgecolor='white')
axes[1].axvline(np.percentile(final_positions, 5), color='red', linestyle='--', label='5% VaR')
axes[1].axvline(final_positions.mean(), color='black', linestyle='-', label='Mean')
axes[1].set_title('Distribution of Final Cash Positions')
axes[1].set_xlabel('Final Cash Position (INR)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print(f"Mean final cash position: ₹{final_positions.mean():,.2f}")
print(f"5% VaR: ₹{np.percentile(final_positions, 5):,.2f}")
print(f"Probability of shortfall: {(final_positions < 0).mean() * 100:.2f}%")
print('Teaching prompt: What liquidity buffer would you hold if this forecast were the base case?')